## 获取__预测结果__从fiftyone__导出__到Excel

### Step 1 — 生成表1（per-image）+ 表1b（per-detection long）

sahi_null_run_whole_rawData_v5_ms2_0605_0923

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple
import re

import numpy as np
import pandas as pd
import fiftyone as fo


# =========================
# Config (single dataset + single pred_field)
# =========================
dataset_name = "sahi_null_run_whole_rawData_v5_ms2_0605_0923"         # TODO
pred_field   = "pred_yolo11n_20pct_null_images_add_rawData_batch_8_final"           # TODO 例如 small_slices_xxx
version      = "sahi_null_run_whole_rawData_v5"       # 你可以改名

# 你可调：用于 high/low 统计的阈值（和训练评估阈值无关）
HIGH_CONF_THR = 0.85
LOW_CONF_THR  = 0.50

out_dir = Path("./_deploy_exports") / version / dataset_name / pred_field
out_dir.mkdir(parents=True, exist_ok=True)

out_img  = out_dir / "table_1_per_image_inference.csv"      # 表1
out_long = out_dir / "table_1b_per_detection_long.csv"       # 表1b


# =========================
# Helpers
# =========================
def parse_dt_focus(fp: str, year=2024) -> Tuple[pd.Timestamp, Optional[int]]:
    """
    解析文件名：MMDD_HHMM_FOCUS.jpg  例如 0729_0606_620.jpg
    返回 (capture_datetime, focus)；失败则 (NaT, None)
    """
    m = re.search(r"(\d{4})_(\d{4})_(\d+)\.(jpg|png)$", Path(fp).name, re.IGNORECASE)
    if not m:
        return pd.NaT, None

    mmdd, hhmm, focus = m.group(1), m.group(2), int(m.group(3))
    return (
        pd.Timestamp(
            year,
            int(mmdd[:2]),
            int(mmdd[2:]),
            int(hhmm[:2]),
            int(hhmm[2:]),
        ),
        focus,
    )

def _safe_dets(sample: fo.Sample, field: str):
    obj = sample.get_field(field)
    if obj is None:
        return []
    dets = getattr(obj, "detections", None)
    return dets if dets else []

def _conf_stats(confs: List[float]) -> Dict[str, float]:
    if not confs:
        return {
            "conf_count": 0,
            "conf_mass": 0.0,
            "conf_min": 0.0,
            "conf_max": 0.0,
            "conf_mean": 0.0,
            "conf_std": 0.0,
            "conf_q05": 0.0,
            "conf_q25": 0.0,
            "conf_q50": 0.0,
            "conf_q75": 0.0,
            "conf_q95": 0.0,
            "conf_iqr": 0.0,
            f"high_conf_count_c{int(HIGH_CONF_THR * 100)}": 0,
            f"high_conf_mass_c{int(HIGH_CONF_THR * 100)}": 0.0,
            f"low_conf_ratio_c{int(LOW_CONF_THR * 100)}": 0.0,
        }

    arr = np.asarray(confs, dtype=float)
    q05, q25, q50, q75, q95 = np.percentile(arr, [5, 25, 50, 75, 95])
    high = arr[arr >= HIGH_CONF_THR]
    low_ratio = float(np.mean(arr < LOW_CONF_THR))

    return {
        "conf_count": int(arr.size),                                                                # 数量
        "conf_mass": float(arr.sum()),                                                              # 质量（confidence 之和）
        "conf_min": float(arr.min()),                                                               # 最小值
        "conf_max": float(arr.max()),                                                               # 最大值
        "conf_mean": float(arr.mean()),                                                             # 平均值
        "conf_std": float(arr.std()),                                                               # 标准差
        "conf_q05": float(q05),                                                                     # 5th percentile
        "conf_q25": float(q25),                                                                     # 25th percentile
        "conf_q50": float(q50),                                                                     # 50th percentile (median)
        "conf_q75": float(q75),                                                                     # 75th percentile
        "conf_q95": float(q95),                                                                     # 95th percentile
        "conf_iqr": float(q75 - q25),                                                               # Interquartile range
        f"high_conf_count_c{int(HIGH_CONF_THR * 100)}": int(high.size),                             # 高置信数量
        f"high_conf_mass_c{int(HIGH_CONF_THR * 100)}": float(high.sum()) if high.size else 0.0,     # 高置信质量
        f"low_conf_ratio_c{int(LOW_CONF_THR * 100)}": low_ratio,                                    # 低置信比例
    }


# =========================
# Run export
# =========================
ds = fo.load_dataset(dataset_name)
print("Loaded:", dataset_name)

if pred_field not in ds.get_field_schema():
    raise ValueError(
        f"pred_field '{pred_field}' not found in dataset '{dataset_name}'.\n"
        f"Available fields: {list(ds.get_field_schema().keys())}"
    )

image_rows: List[Dict[str, Any]] = []
det_rows: List[Dict[str, Any]] = []

for s in ds.iter_samples(progress=True):
    dets = _safe_dets(s, pred_field)
    confs = [d.confidence for d in dets if d.confidence is not None]

    capture_dt, focus = parse_dt_focus(s.filepath, year=2024)
    date = None if pd.isna(capture_dt) else capture_dt.date()
    time = None if pd.isna(capture_dt) else capture_dt.time()

    stats = _conf_stats(confs)

    # -------- 表1：per-image（一行一图）--------
    image_rows.append({
        "dataset_name": dataset_name,
        "pred_field": pred_field,
        "sample_id": str(s.id),
        "filepath": s.filepath,

        "datetime": capture_dt,
        "date": date,
        "time": time,
        "focus": focus,

        "pred_count": len(dets),

        # 分布统计（支持 box plot）
        **stats,

        # raw list（你选 C）
        "confidences_list": confs,
    })

    # -------- 表1b：per-detection long（一行一个 detection）--------
    for i, d in enumerate(dets):
        bb = getattr(d, "bounding_box", None)  # [x, y, w, h] in relative coords (0..1)
        if bb and len(bb) == 4:
            bx, by, bw, bh = bb
            area = float(bw * bh)
        else:
            bx = by = bw = bh = np.nan
            area = np.nan

        det_rows.append({
            "dataset_name": dataset_name,
            "pred_field": pred_field,
            "sample_id": str(s.id),
            "filepath": s.filepath,

            "datetime": capture_dt,
            "date": date,
            "time": time,
            "focus": focus,

            "det_idx": i,
            "confidence": getattr(d, "confidence", np.nan),

            "bbox_x": bx,
            "bbox_y": by,
            "bbox_w": bw,
            "bbox_h": bh,
            "bbox_area": area,
        })

# 写出 CSV
img_df = pd.DataFrame(image_rows)
det_df = pd.DataFrame(det_rows)

img_df.to_csv(out_img, index=False)
det_df.to_csv(out_long, index=False)

print("[SAVE]", out_img, "shape=", img_df.shape)
print("[SAVE]", out_long, "shape=", det_df.shape)



Loaded: sahi_null_run_whole_rawData_v5_ms2_0605_0923
 100% |█████████████| 12254/12254 [7.2s elapsed, 0s remaining, 3.8K samples/s]        
[SAVE] _deploy_exports/sahi_null_run_whole_rawData_v5/sahi_null_run_whole_rawData_v5_ms2_0605_0923/pred_yolo11n_20pct_null_images_add_rawData_batch_8_final/table_1_per_image_inference.csv shape= (12254, 25)
[SAVE] _deploy_exports/sahi_null_run_whole_rawData_v5/sahi_null_run_whole_rawData_v5_ms2_0605_0923/pred_yolo11n_20pct_null_images_add_rawData_batch_8_final/table_1b_per_detection_long.csv shape= (12609, 15)


In [14]:
# table_1_per_image_inference.csv 
img_df

,dataset_name,pred_field,sample_id,filepath,datetime,date,time,focus,pred_count,conf_count,...,conf_q05,conf_q25,conf_q50,conf_q75,conf_q95,conf_iqr,high_conf_count_c85,high_conf_mass_c85,low_conf_ratio_c50,confidences_list
0,sahi_null_run_whole_rawData_v5_ms2_0605_0923,pred_yolo11n_20pct_null_images_add_rawData_bat...,6986c0f4a96fca8fdc57ac8c,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-06-05 11:33:00,2024-06-05,11:33:00,700,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,[]
1,sahi_null_run_whole_rawData_v5_ms2_0605_0923,pred_yolo11n_20pct_null_images_add_rawData_bat...,6986c0f4a96fca8fdc57ac8d,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-06-05 12:01:00,2024-06-05,12:01:00,700,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,[]
2,sahi_null_run_whole_rawData_v5_ms2_0605_0923,pred_yolo11n_20pct_null_images_add_rawData_bat...,6986c0f4a96fca8fdc57ac8e,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-06-05 12:31:00,2024-06-05,12:31:00,700,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,[]
3,sahi_null_run_whole_rawData_v5_ms2_0605_0923,pred_yolo11n_20pct_null_images_add_rawData_bat...,6986c0f4a96fca8fdc57ac8f,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-06-05 13:01:00,2024-06-05,13:01:00,700,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,[]
4,sahi_null_run_whole_rawData_v5_ms2_0605_0923,pred_yolo11n_20pct_null_images_add_rawData_bat...,6986c0f4a96fca8fdc57ac90,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-06-05 13:31:00,2024-06-05,13:31:00,700,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12249,sahi_null_run_whole_rawData_v5_ms2_0605_0923,pred_yolo11n_20pct_null_images_add_rawData_bat...,6986c0f5a96fca8fdc57dc65,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-09-23 19:21:00,2024-09-23,19:21:00,540,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,[]
12250,sahi_null_run_whole_rawData_v5_ms2_0605_0923,pred_yolo11n_20pct_null_images_add_rawData_bat...,6986c0f5a96fca8fdc57dc66,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-09-23 19:23:00,2024-09-23,19:23:00,580,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,[]
12251,sahi_null_run_whole_rawData_v5_ms2_0605_0923,pred_yolo11n_20pct_null_images_add_rawData_bat...,6986c0f5a96fca8fdc57dc67,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-09-23 19:24:00,2024-09-23,19:24:00,600,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,[]
12252,sahi_null_run_whole_rawData_v5_ms2_0605_0923,pred_yolo11n_20pct_null_images_add_rawData_bat...,6986c0f5a96fca8fdc57dc68,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-09-23 19:25:00,2024-09-23,19:25:00,640,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,[]


In [17]:
from datetime import datetime

df_sub = img_df.copy()

df_sub = df_sub.drop(columns=['dataset_name', 'pred_field', 'sample_id'])
df_sub = df_sub[
    (
        df_sub["datetime"]
        > datetime.strptime("2024-07-16T23:24:41.829Z", "%Y-%m-%dT%H:%M:%S.%fZ")
    )
    & (
        df_sub["datetime"]
        < datetime.strptime("2024-07-30T23:25:11.424Z", "%Y-%m-%dT%H:%M:%S.%fZ")
    )
]
df_sub

,filepath,datetime,date,time,focus,pred_count,conf_count,conf_mass,conf_min,conf_max,...,conf_q05,conf_q25,conf_q50,conf_q75,conf_q95,conf_iqr,high_conf_count_c85,high_conf_mass_c85,low_conf_ratio_c50,confidences_list
2624,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-07-17 06:03:00,2024-07-17,06:03:00,540,0,0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,0.000000,0.0,[]
2625,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-07-17 06:04:00,2024-07-17,06:04:00,580,1,1,0.877321,0.877321,0.877321,...,0.877321,0.877321,0.877321,0.877321,0.877321,0.000000,1,0.877321,0.0,[0.8773210644721985]
2626,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-07-17 06:05:00,2024-07-17,06:05:00,600,3,3,2.610990,0.866961,0.874793,...,0.867188,0.868098,0.869236,0.872014,0.874238,0.003916,3,2.610990,0.0,"[0.874793291091919, 0.8692356944084167, 0.8669..."
2627,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-07-17 06:07:00,2024-07-17,06:07:00,640,3,3,2.678902,0.868964,0.911282,...,0.871933,0.883810,0.898656,0.904969,0.910019,0.021159,3,2.678902,0.0,"[0.9112818241119385, 0.8986560106277466, 0.868..."
2628,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-07-17 06:08:00,2024-07-17,06:08:00,680,2,2,1.804691,0.889815,0.914875,...,0.891068,0.896080,0.902345,0.908610,0.913622,0.012530,2,1.804691,0.0,"[0.9148751497268677, 0.8898154497146606]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4607,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-07-30 20:31:00,2024-07-30,20:31:00,540,0,0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,0.000000,0.0,[]
4608,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-07-30 20:33:00,2024-07-30,20:33:00,580,0,0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,0.000000,0.0,[]
4609,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-07-30 20:34:00,2024-07-30,20:34:00,600,0,0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,0.000000,0.0,[]
4610,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-07-30 20:35:00,2024-07-30,20:35:00,640,0,0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,0.000000,0.0,[]


In [3]:
# table_1b_per_detection_long.csv
det_df

,dataset_name,pred_field,sample_id,filepath,capture_datetime,capture_date,capture_time,hour,doy,focus,det_idx,confidence,bbox_x,bbox_y,bbox_w,bbox_h,bbox_area
0,sahi_null_run_whole_rawData_v5_ms2_0605_0923,pred_yolo11n_20pct_null_images_add_rawData_bat...,6986c0f4a96fca8fdc57aca5,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-06-06 09:01:00,2024-06-06,09:01:00,9,158,700,0,0.852486,0.313925,0.601069,0.009828,0.012742,0.000125
1,sahi_null_run_whole_rawData_v5_ms2_0605_0923,pred_yolo11n_20pct_null_images_add_rawData_bat...,6986c0f4a96fca8fdc57acbc,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-06-06 20:31:00,2024-06-06,20:31:00,20,158,700,0,0.865482,0.727783,0.631985,0.012545,0.019025,0.000239
2,sahi_null_run_whole_rawData_v5_ms2_0605_0923,pred_yolo11n_20pct_null_images_add_rawData_bat...,6986c0f4a96fca8fdc57acc0,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-06-07 07:31:00,2024-06-07,07:31:00,7,159,700,0,0.922013,0.630475,0.549642,0.011237,0.015872,0.000178
3,sahi_null_run_whole_rawData_v5_ms2_0605_0923,pred_yolo11n_20pct_null_images_add_rawData_bat...,6986c0f4a96fca8fdc57acc0,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-06-07 07:31:00,2024-06-07,07:31:00,7,159,700,1,0.893447,0.206773,0.381043,0.011949,0.015006,0.000179
4,sahi_null_run_whole_rawData_v5_ms2_0605_0923,pred_yolo11n_20pct_null_images_add_rawData_bat...,6986c0f4a96fca8fdc57acc0,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-06-07 07:31:00,2024-06-07,07:31:00,7,159,700,2,0.891570,0.439936,0.606471,0.012440,0.015912,0.000198
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12604,sahi_null_run_whole_rawData_v5_ms2_0605_0923,pred_yolo11n_20pct_null_images_add_rawData_bat...,6986c0f5a96fca8fdc57dc47,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-09-23 16:21:00,2024-09-23,16:21:00,16,267,540,0,0.856718,0.416027,0.789204,0.011431,0.015149,0.000173
12605,sahi_null_run_whole_rawData_v5_ms2_0605_0923,pred_yolo11n_20pct_null_images_add_rawData_bat...,6986c0f5a96fca8fdc57dc4c,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-09-23 16:51:00,2024-09-23,16:51:00,16,267,540,0,0.860893,0.416140,0.789372,0.011066,0.015024,0.000166
12606,sahi_null_run_whole_rawData_v5_ms2_0605_0923,pred_yolo11n_20pct_null_images_add_rawData_bat...,6986c0f5a96fca8fdc57dc51,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-09-23 17:21:00,2024-09-23,17:21:00,17,267,540,0,0.865853,0.416049,0.788399,0.011077,0.015909,0.000176
12607,sahi_null_run_whole_rawData_v5_ms2_0605_0923,pred_yolo11n_20pct_null_images_add_rawData_bat...,6986c0f5a96fca8fdc57dc56,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...,2024-09-23 17:51:00,2024-09-23,17:51:00,17,267,540,0,0.889217,0.416090,0.788641,0.010833,0.015601,0.000169


### Step 2 — 表2：Per timepoint × focus（每个 datetime×focus 一行）

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

# ==============
# Config
# ==============
in_img = Path("./_deploy_exports/sahi_null_run_whole_rawData_v5/sahi_null_run_whole_rawData_v5_ms2_0605_0923/pred_yolo11n_20pct_null_images_add_rawData_batch_8_final/table_1_per_image_inference.csv")
out_dir = in_img.parent
out_txf = out_dir / "table_2_per_timepoint_focus.csv"   # 表2

# ==============
# Load
# ==============
df = pd.read_csv(in_img)

# 关键：把 datetime 读回来
df["capture_datetime"] = pd.to_datetime(df["capture_datetime"], errors="coerce")

# focus 可能有空
# df["focus"] = pd.to_numeric(df["focus"], errors="coerce").astype("Int64")

# presence
df["has_detection"] = (df["pred_count"].fillna(0) > 0).astype(int)

gcols = ["capture_datetime", "focus"]

agg = df.groupby(gcols, as_index=False).agg(
    # time helpers（如果同组一致，取 first 就行）
    capture_date=("capture_date", "first"),
    capture_time=("capture_time", "first"),
    hour=("hour", "first"),
    doy=("doy", "first"),

    # frames
    frame_count=("filepath", "count"),

    # counts
    total_pred_count=("pred_count", "sum"),
    mean_pred_count=("pred_count", "mean"),
    median_pred_count=("pred_count", "median"),
    std_pred_count=("pred_count", "std"),

    # confidence mass (表1字段名是 conf_mass)
    confidence_mass_sum=("conf_mass", "sum"),
    confidence_mass_mean=("conf_mass", "mean"),
    confidence_mass_std=("conf_mass", "std"),

    # presence
    frames_with_detection=("has_detection", "sum"),
)

# detection rate
agg["detection_rate"] = agg["frames_with_detection"] / agg["frame_count"]

# 聚合 “每帧的分布统计”（可用于 focus 稳定性）
# 这里用 mean 作为 timepoint×focus 的概况
extra = df.groupby(gcols, as_index=False).agg(
    conf_q50_time_mean=("conf_q50", "mean"),
    conf_iqr_time_mean=("conf_iqr", "mean"),
    conf_std_time_mean=("conf_std", "mean"),
)

agg = agg.merge(extra, on=gcols, how="left")

# std 可能是 NaN（组内只有1帧），可按需填 0
for c in ["std_pred_count", "confidence_mass_std"]:
    agg[c] = agg[c].fillna(0.0)

agg.to_csv(out_txf, index=False)
print("[SAVE]", out_txf, "shape=", agg.shape)


[SAVE] _deploy_exports/sahi_null_run_whole_rawData_v5/sahi_null_run_whole_rawData_v5_ms2_0605_0923/pred_yolo11n_20pct_null_images_add_rawData_batch_8_final/table_2_per_timepoint_focus.csv shape= (12254, 19)


In [5]:
# table_2_per_timepoint_focus.csv
agg

,capture_datetime,focus,capture_date,capture_time,hour,doy,frame_count,total_pred_count,mean_pred_count,median_pred_count,std_pred_count,confidence_mass_sum,confidence_mass_mean,confidence_mass_std,frames_with_detection,detection_rate,conf_q50_time_mean,conf_iqr_time_mean,conf_std_time_mean
0,2024-06-05 11:33:00,700,2024-06-05,11:33:00,11,157,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
1,2024-06-05 12:01:00,700,2024-06-05,12:01:00,12,157,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
2,2024-06-05 12:31:00,700,2024-06-05,12:31:00,12,157,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
3,2024-06-05 13:01:00,700,2024-06-05,13:01:00,13,157,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
4,2024-06-05 13:31:00,700,2024-06-05,13:31:00,13,157,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12249,2024-09-23 19:21:00,540,2024-09-23,19:21:00,19,267,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
12250,2024-09-23 19:23:00,580,2024-09-23,19:23:00,19,267,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
12251,2024-09-23 19:24:00,600,2024-09-23,19:24:00,19,267,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
12252,2024-09-23 19:25:00,640,2024-09-23,19:25:00,19,267,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0


### 表3

In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

# ==============
# Config
# ==============
in_txf = Path("./_deploy_exports/sahi_null_run_whole_rawData_v5/sahi_null_run_whole_rawData_v5_ms2_0605_0923/pred_yolo11n_20pct_null_images_add_rawData_batch_8_final/table_2_per_timepoint_focus.csv")
out_dir = in_txf.parent
out_fused = out_dir / "table_3_per_timepoint_fused_max.csv"   # 表3

# ==============
# Load
# ==============
df = pd.read_csv(in_txf)
df["capture_datetime"] = pd.to_datetime(df["capture_datetime"], errors="coerce")

# 关键列检查
need_cols = ["capture_datetime", "focus", "confidence_mass_sum", "total_pred_count", "detection_rate"]
missing = [c for c in need_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in table2: {missing}")

# ==============
# Fuse by MAX(confidence_mass_sum)
# ==============
rows = []
for dt, g in df.groupby("capture_datetime", sort=True):
    if pd.isna(dt):
        continue

    # 选取 confidence_mass_sum 最大的那一行（若并列，取第一条）
    idx = g["confidence_mass_sum"].astype(float).idxmax()
    best = df.loc[idx]

    rows.append({
        # time
        "capture_datetime": dt,
        "capture_date": best.get("capture_date", None),
        "capture_time": best.get("capture_time", None),
        "hour": best.get("hour", None),
        "doy": best.get("doy", None),

        # fused outputs (MAX)
        "confidence_mass_max": float(best["confidence_mass_sum"]),
        "pred_count_max": int(best["total_pred_count"]),
        "detection_rate_max": float(best["detection_rate"]),

        # which focus won
        "focus_selected": best["focus"],

        # support info
        "num_focus_available": int(g["focus"].nunique(dropna=True)),
        "num_focus_detected": int((g["frames_with_detection"] > 0).sum()) if "frames_with_detection" in g.columns else int((g["detection_rate"] > 0).sum()),
        "any_focus_detected": int((g["detection_rate"] > 0).any()),

        # across-focus disagreement (optional but useful)
        "confidence_mass_range_across_focus": float(g["confidence_mass_sum"].max() - g["confidence_mass_sum"].min()),
        "confidence_mass_std_across_focus": float(g["confidence_mass_sum"].std(ddof=0)) if len(g) > 1 else 0.0,
    })

out = pd.DataFrame(rows).sort_values("capture_datetime")
out.to_csv(out_fused, index=False)
print("[SAVE]", out_fused, "shape=", out.shape)


[SAVE] _deploy_exports/sahi_null_run_whole_rawData_v5/sahi_null_run_whole_rawData_v5_ms2_0605_0923/pred_yolo11n_20pct_null_images_add_rawData_batch_8_final/table_3_per_timepoint_fused_max.csv shape= (12254, 14)


In [7]:
# table_3_per_timepoint_fused_max.csv
out

,capture_datetime,capture_date,capture_time,hour,doy,confidence_mass_max,pred_count_max,detection_rate_max,focus_selected,num_focus_available,num_focus_detected,any_focus_detected,confidence_mass_range_across_focus,confidence_mass_std_across_focus
0,2024-06-05 11:33:00,2024-06-05,11:33:00,11,157,0.0,0,0.0,700,1,0,0,0.0,0.0
1,2024-06-05 12:01:00,2024-06-05,12:01:00,12,157,0.0,0,0.0,700,1,0,0,0.0,0.0
2,2024-06-05 12:31:00,2024-06-05,12:31:00,12,157,0.0,0,0.0,700,1,0,0,0.0,0.0
3,2024-06-05 13:01:00,2024-06-05,13:01:00,13,157,0.0,0,0.0,700,1,0,0,0.0,0.0
4,2024-06-05 13:31:00,2024-06-05,13:31:00,13,157,0.0,0,0.0,700,1,0,0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12249,2024-09-23 19:21:00,2024-09-23,19:21:00,19,267,0.0,0,0.0,540,1,0,0,0.0,0.0
12250,2024-09-23 19:23:00,2024-09-23,19:23:00,19,267,0.0,0,0.0,580,1,0,0,0.0,0.0
12251,2024-09-23 19:24:00,2024-09-23,19:24:00,19,267,0.0,0,0.0,600,1,0,0,0.0,0.0
12252,2024-09-23 19:25:00,2024-09-23,19:25:00,19,267,0.0,0,0.0,640,1,0,0,0.0,0.0


# Data Visualization with Plotly
This section visualizes the aggregated data using Plotly for interactive plots.

In [9]:
import plotly.express as px

# Plot 1: Total Prediction Count Over Time
fig1 = px.line(agg, x='capture_datetime', y='total_pred_count', title='Total Prediction Count Over Time')
fig1.show()

# Plot 2: Mean Prediction Count by Focus
fig2 = px.bar(agg, x='focus', y='mean_pred_count', title='Mean Prediction Count by Focus')
fig2.show()

# Plot 3: Detection Rate Distribution
fig3 = px.histogram(agg, x='detection_rate', nbins=30, title='Detection Rate Distribution')
fig3.show()